# `semantic.v_beat_rate_by_cut` — view

Thin view over Gold. No logic beyond shaping.

A view is its own definition, so there is no load step and no etl task to pair with this one.

In [ ]:
-- THE FOUR CUTS. One view, one extra column saying which cut a row belongs to, because
-- four near-identical views would be four things to maintain and explain.
--
-- Three of them are a GROUP BY over a column dim_ticker already holds. The fourth goes
-- through the bridge, and is the one place a trust can be counted more than once.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_beat_rate_by_cut
COMMENT 'Beat rate by management group, by manager, by sole/multi manager, and by AIC sector'
AS
WITH trusts AS (
  SELECT f.ticker_key, f.management_group_key,
         f.horizon_years, f.beat_index, f.total_return, f.volatility,
         f.risk_adjusted_return,
         d.manager_structure, d.aic_sector
  FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance f
  JOIN `index-vs-trust-pipeline`.gold.dim_ticker d
    ON d.ticker_key = f.ticker_key AND d.entity_type = 'Trust'
),
cut AS (
  -- The house now comes from its own dimension rather than a string on dim_ticker. Joined
  -- here and not in `trusts`, so this branch cannot take the other three down with it.
  SELECT 'management group' AS cut_by, g.management_group AS cut_value,
         t.horizon_years, t.beat_index, t.total_return, t.volatility, t.risk_adjusted_return
  FROM trusts t
  JOIN `index-vs-trust-pipeline`.gold.dim_management_group g
    ON g.management_group_key = t.management_group_key
  UNION ALL
  -- IMPACT BASIS. A trust with six managers is counted six times, once for each of them.
  -- That is right for "what share of managers had a trust that beat the index" and wrong
  -- for anything that has to add up to the 440 rows in the fact -- use the bridge's
  -- allocation_factor for that.
  SELECT 'manager', m.manager_name,
         t.horizon_years, t.beat_index, t.total_return, t.volatility, t.risk_adjusted_return
  FROM trusts t
  JOIN `index-vs-trust-pipeline`.gold.bridge_ticker_manager b ON b.ticker_key = t.ticker_key
  JOIN `index-vs-trust-pipeline`.gold.dim_manager m           ON m.manager_key = b.manager_key
  UNION ALL
  SELECT 'manager structure', t.manager_structure,
         t.horizon_years, t.beat_index, t.total_return, t.volatility, t.risk_adjusted_return
  FROM trusts t
  UNION ALL
  SELECT 'aic sector', t.aic_sector,
         t.horizon_years, t.beat_index, t.total_return, t.volatility, t.risk_adjusted_return
  FROM trusts t
)
SELECT cut_by,
       cut_value,
       horizon_years,
       COUNT(*)                                                   AS trusts,
       SUM(CASE WHEN beat_index THEN 1 ELSE 0 END)                AS beat_count,
       ROUND(100.0 * SUM(CASE WHEN beat_index THEN 1 ELSE 0 END)
             / COUNT(*), 1)                                       AS beat_rate_pct,
       ROUND(100 * PERCENTILE_APPROX(total_return, 0.5), 1)       AS median_return_pct,
       ROUND(100 * PERCENTILE_APPROX(volatility, 0.5), 1)         AS median_volatility_pct,
       ROUND(PERCENTILE_APPROX(risk_adjusted_return, 0.5), 2)     AS median_risk_adjusted
FROM cut
GROUP BY cut_by, cut_value, horizon_years;

## Verification

Expected: four values in `cut_by`, and the `manager` cut carrying more trust-rows than the
others because a multi-manager trust is counted once per manager. On the `management group`
cut every horizon must still total the same number of trusts as `v_beat_rate` — if it does
not, `management_group_key` has not been filled and the join is dropping rows.

In [ ]:
SELECT cut_by,
       COUNT(*)              AS rows_in_cut,
       COUNT(DISTINCT cut_value) AS distinct_values,
       SUM(trusts)           AS trust_rows_counted
FROM `index-vs-trust-pipeline`.semantic.v_beat_rate_by_cut
GROUP BY cut_by
ORDER BY cut_by;